# 15 · 2축 휨 간략식과 엄밀해 비교

`KDS.biaxial_bending_diagram` 은 상관면을 엄밀하게 계산한다. 여기서는
실무에서 널리 쓰이는 Bresler 간략식과 비교한다.

간략식은 KDS 14 20 의 조문이 아니라 문헌에서 인정되는 근사법이다.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [2]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS


def beam_section(fck=27, fy=400):
    """400 x 600 보 단면 (상부 2-D16, 하부 4-D22, 피복 50 mm)."""
    kds = KDS(column_type="tie")
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=600, b=400,
        dia_top=16, area_top=198.6, n_top=2, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=4, c_bot=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec


def column_section(fck=27, fy=400, column_type="tie"):
    """500 x 500 기둥 단면 (8-D22, 피복 50 mm)."""
    kds = KDS(column_type=column_type)
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=500, b=500,
        dia_top=22, area_top=387.1, n_top=3, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=3, c_bot=50,
        dia_side=22, area_side=387.1, n_side=1, c_side=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec

In [3]:
from concreteproperties_kds.biaxial import (
    check_bresler_reciprocal,
    check_load_contour,
    compare_with_exact,
)

N_DESIGN, M_UX, M_UY = 1200e3, 200e6, 120e6

kds, _ = column_section()

f_x, _, phi_x = kds.ultimate_bending_capacity(theta=0, n_design=N_DESIGN)
f_y, _, phi_y = kds.ultimate_bending_capacity(
    theta=-np.pi / 2, n_design=N_DESIGN
)

phi_m_nx, phi_m_ny = abs(f_x.m_x), abs(f_y.m_y)

print(f"Nd = {N_DESIGN / 1e3:,.0f} kN")
print(f"x 축  phi*Mnx = {phi_m_nx / 1e6:8.2f} kN.m  (phi = {phi_x:.3f})")
print(f"y 축  phi*Mny = {phi_m_ny / 1e6:8.2f} kN.m  (phi = {phi_y:.3f})")
print(f"소요  Mux = {M_UX / 1e6:.1f},  Muy = {M_UY / 1e6:.1f} kN.m")

Nd = 1,200 kN
x 축  phi*Mnx =   386.23 kN.m  (phi = 0.828)
y 축  phi*Mny =   386.23 kN.m  (phi = 0.828)
소요  Mux = 200.0,  Muy = 120.0 kN.m


In [4]:
check_load_contour(
    m_ux=M_UX, m_uy=M_UY,
    phi_m_nx=phi_m_nx, phi_m_ny=phi_m_ny, alpha=1.0,
).print_results()

2축 휨 검토 - 등하중선법 (alpha = 1.0)
소요                =         0.8285
강도                =         1.0000
소요/강도           =         0.8285
판정                =             만족
비고 : alpha = 1.0 은 보수측, 2.0 은 비보수측에 가깝다.


## 엄밀 상관면과 비교

In [5]:
f_bb, _ = kds.biaxial_bending_diagram(
    n_design=N_DESIGN, n_points=48, progress_bar=False
)
m_x = np.array([r.m_x for r in f_bb.results])
m_y = np.array([r.m_y for r in f_bb.results])

target = np.arctan2(M_UY, M_UX)
angles = np.arctan2(m_y, m_x)
idx = int(np.argmin(np.abs(np.angle(np.exp(1j * (angles - target))))))

exact = float(np.hypot(M_UX, M_UY)) / float(np.hypot(m_x[idx], m_y[idx]))
print(f"엄밀해 소요/강도 = {exact:.4f}")
print()
print(f"{'alpha':>8} {'등하중선법':>12} {'보수적':>8}")
print("-" * 32)
for alpha, value, conservative in compare_with_exact(
    m_ux=M_UX, m_uy=M_UY,
    phi_m_nx=phi_m_nx, phi_m_ny=phi_m_ny, exact_ratio=exact,
):
    print(f"{alpha:8.2f} {value:12.4f} {'예' if conservative else '아니오':>8}")

엄밀해 소요/강도 = 0.7536

   alpha        등하중선법      보수적
--------------------------------
    1.00       0.8285        예
    1.25       0.6712      아니오
    1.50       0.5458      아니오
    2.00       0.3647      아니오


In [6]:
fig, ax = plt.subplots(figsize=(6.4, 5.6))
ax.plot(m_x / 1e6, m_y / 1e6, "k-", lw=1.6, label="exact (KDS analysis)")

mx = np.linspace(0, phi_m_nx, 200)
for alpha in [1.0, 1.25, 1.5, 2.0]:
    inner = np.clip(1 - (mx / phi_m_nx) ** alpha, 0, None)
    ax.plot(
        mx / 1e6, phi_m_ny * inner ** (1 / alpha) / 1e6,
        "--", lw=1, label=f"contour alpha = {alpha:.2f}",
    )

ax.plot(M_UX / 1e6, M_UY / 1e6, "r*", ms=13, label="demand")
ax.set_xlim(0, phi_m_nx / 1e6 * 1.05)
ax.set_ylim(0, phi_m_ny / 1e6 * 1.05)
ax.set_xlabel("phi*Mx (kN.m)")
ax.set_ylabel("phi*My (kN.m)")
ax.set_title(f"Exact vs load contour, Nd = {N_DESIGN / 1e3:,.0f} kN")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

엄밀 상관면(검은 실선)은 $\alpha = 1.0$ 직선보다 바깥, $\alpha = 1.25$
곡선보다 안쪽에 있다. 즉 **$\alpha = 1.0$ 만 보수적**이고 그 이상은
위험측이다. $\alpha$ 를 임의로 키우지 말고 엄밀해로 확인하는 편이 낫다.

## Bresler 역하중법

In [7]:
n_max_nom, n_max_des = kds.max_axial_strength()
f_mi, _, _ = kds.moment_interaction_diagram(
    theta=0, n_points=32, progress_bar=False
)
n_list = np.array([r.n for r in f_mi.results])
m_list = np.array([r.m_x for r in f_mi.results])


def axial_capacity_at_eccentricity(e):
    residual = m_list - n_list * e
    for i in range(len(residual) - 1):
        if residual[i] * residual[i + 1] <= 0:
            t = residual[i] / (residual[i] - residual[i + 1])
            return float(n_list[i] + t * (n_list[i + 1] - n_list[i]))
    return float(n_list[0])


check_bresler_reciprocal(
    p_u=N_DESIGN,
    phi_p_nx=axial_capacity_at_eccentricity(M_UX / N_DESIGN),
    phi_p_ny=axial_capacity_at_eccentricity(M_UY / N_DESIGN),
    phi_p_o=n_max_des,
    fck=27, a_g=500.0 * 500.0,
).print_results()

2축 휨 검토 - Bresler 역하중법
소요                =   1200000.0000
강도                =   1684990.2870
소요/강도           =         0.7122
판정                =             만족
